In [1]:
import GtoTmodel as GtoTmodel
import Circuits as Circuits 
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
circuits = Circuits.CircuitS()


Graph 4['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 6['VDD', 'VSS', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'R1', 'R1_P', 'R1_N', 'C1', 'C1_P', 'C1_N']
Graph 9['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 14['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'R1', 'R1_P', 'R1_N']
Graph 17['VDD', 'VSS', 'VIN1', 'VOUT1', 'R1', 'R1_P', 'R1_N', 'R2', 'R2_P', 'R2_N', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B']
Graph 20['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B', 'R1', 'R1_P', 'R1_N']
Graph 22['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'PM2', 'PM2_D', 'PM2_G', 'PM2_S', 'PM2_B']
Graph 24['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_

In [3]:
# Assuming circuits has a method or attribute to get the matrix, e.g., circuits.get_matrix()
matrix = circuits.component_lists  # Replace with the actual method or attribute
max_length = max(len(vector) for vector in matrix)
print("Maximum length of vectors in the matrix:", max_length)

Maximum length of vectors in the matrix: 310


In [4]:
circuits.vocab.__len__()  # This should give the number of components

892

In [5]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Load

In [7]:
num_ciruits = 320
graph_dataset =  circuits.graphs
text_dataset = circuits.component_indices

# Convert graph_dataset and text_dataset to numpy arrays
graph_dataset = np.array(graph_dataset, dtype=object)
text_dataset = np.array(text_dataset, dtype=object)

# Convert numpy arrays to PyTorch tensors
graph_dataset = [torch.tensor(graph, dtype=torch.float32) for graph in graph_dataset]
graph_dataset = [torch.cat((torch.nn.functional.pad(graph, (0, max_length - graph.size(0))),torch.tensor([[9]*310]))) for graph in graph_dataset]
text_dataset = [torch.tensor(text+[893], dtype=torch.int) for text in text_dataset]

In [8]:
graph_dataset[1]
text_dataset[1]

tensor([384, 637, 538, 492, 624, 284, 719, 878, 295, 180,  97, 792, 694, 498,
        893], dtype=torch.int32)

In [9]:
graph_dataset = torch.cat(graph_dataset).to(device)
text_dataset = torch.cat(text_dataset,).to(device)

In [10]:
graph_dataset.shape, text_dataset.shape

(torch.Size([351461, 310]), torch.Size([351461]))

### Generate Batchers

In [11]:
batch_size = 4
block_size = 16  # Length of each sequence block


def get_batch( batch_size=4, block_size=16):
    start_indices = torch.randint(0, len(graph_dataset) - block_size, (batch_size,))
    """
    Get a batch of sequences from the text_dataset.

    Args:
        text_dataset (torch.Tensor): The dataset containing text sequences.
        batch_size (int): The number of sequences in the batch.
        block_size (int): The length of each sequence block.

    Returns:
        torch.Tensor: A batch of sequences with shape (batch_size, block_size).
    """
    # Combine graph data and text data for the batch
    
    batch_graph = torch.stack([graph_dataset[i:i + block_size] for i in start_indices])
    batch_text = torch.stack([text_dataset[i:i + block_size] for i in start_indices])
    # Extract sequences of length block_size starting from the sampled indices

    return batch_text, batch_graph 




### Importing the model

In [12]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
batch_text = batch_text.to(device)
batch_graph = batch_graph.to(device)


c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [13]:
learning_rate = 0.001
num_epochs = 10

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training

In [14]:
batch_text, batch_graph = get_batch(batch_size=4, block_size=16)
batch_text = batch_text.to(device)
batch_graph = batch_graph.to(device)

In [17]:

# model.to('cpu')
# atch_text = batch_text.to('cpu')
# batch_graph = batch_graph.to('cpu')
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
for _ in range(10000):
    batch_text, batch_graph = get_batch(batch_size=4, block_size=16)
    batch_text = batch_text.to(device)
    batch_graph = batch_graph.to(device)
    # Forward pass through the model
    # Ensure batch_text has at least two dimensions
    if batch_text.dim() == 1:
        batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

    # Ensure batch_graph has at least two dimensions
    if batch_graph.dim() == 1:
        batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

    # Forward pass through the model
    output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

    print("Output shape:", output.shape)  # Should be (batch_size, seq_length, vocab_size)

    # Calculate loss
    loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
    print("Loss:", loss.item())  # Print the loss value
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # Print the updated model parameter

Output shape: torch.Size([4, 15, 894])
Loss: 0.13784687221050262
Output shape: torch.Size([4, 15, 894])
Loss: 0.7988790273666382
Output shape: torch.Size([4, 15, 894])
Loss: 0.3264114558696747
Output shape: torch.Size([4, 15, 894])
Loss: 0.3647860288619995
Output shape: torch.Size([4, 15, 894])
Loss: 0.2534381151199341
Output shape: torch.Size([4, 15, 894])
Loss: 0.19367381930351257
Output shape: torch.Size([4, 15, 894])
Loss: 0.19893351197242737
Output shape: torch.Size([4, 15, 894])
Loss: 0.20247481763362885
Output shape: torch.Size([4, 15, 894])
Loss: 0.16955938935279846
Output shape: torch.Size([4, 15, 894])
Loss: 0.6512728929519653
Output shape: torch.Size([4, 15, 894])
Loss: 0.2208576798439026
Output shape: torch.Size([4, 15, 894])
Loss: 0.43442821502685547
Output shape: torch.Size([4, 15, 894])
Loss: 0.3111266493797302
Output shape: torch.Size([4, 15, 894])
Loss: 0.6824844479560852
Output shape: torch.Size([4, 15, 894])
Loss: 0.14219148457050323
Output shape: torch.Size([4, 15, 

KeyboardInterrupt: 

In [ ]:
# # Get a batch of data


# # Move the batch to the appropriate device

# batch_text, batch_graph = get_batch(batch_size=4, block_size=16)
# batch_text = batch_text.to(device)
# batch_graph = batch_graph.to(device)
# # Forward pass through the model
# # Ensure batch_text has at least two dimensions
# if batch_text.dim() == 1:
#     batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

# # Ensure batch_graph has at least two dimensions
# if batch_graph.dim() == 1:
#     batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

# # Forward pass through the model
# output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

# print("Output shape:", output.shape)  # Should be (batch_size, seq_length, vocab_size)

# # Calculate loss
# loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
# print("Loss:", loss.item())  # Print the loss value
# # Backward pass and optimization
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()
# # Print the updated model parameters
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name, param.data)

Output shape: torch.Size([4, 15, 894])
Loss: 0.30624788999557495
encoder_embedding.weight tensor([[ 4.7667e-01,  7.3090e-02, -9.7593e-02,  ..., -1.2547e-01,
         -8.1783e-02, -1.2360e-01],
        [ 1.2092e-01,  3.1624e-01, -3.5804e-02,  ..., -4.2609e-02,
          2.9857e-03, -3.2171e-02],
        [ 5.3750e-02,  1.8167e-01,  3.8403e-02,  ...,  9.5403e-02,
          1.0948e-01,  9.5721e-02],
        ...,
        [ 3.9544e-02,  3.3617e-01, -2.8730e-02,  ...,  3.0780e-02,
          5.5433e-02,  9.3463e-02],
        [-3.4919e-01, -1.7273e-02,  8.2634e-02,  ..., -6.3906e-03,
          9.4151e-02,  1.0017e-01],
        [ 3.0898e-01, -1.0032e-01, -4.2557e-01,  ..., -9.0918e-03,
         -2.3514e-02,  3.2942e-04]], device='cuda:0')
encoder_embedding.bias tensor([-0.1354, -0.1017, -0.0805,  0.1018, -0.0407, -0.0851,  0.0483,  0.0794,
        -0.0188,  0.0156, -0.0322, -0.0350,  0.1081, -0.0124,  0.0378, -0.0646],
       device='cuda:0')
decoder_embedding.weight tensor([[-0.6652, -0.9367,  